### Guardrails with LangChain

This notebook covers everything you need to know about implementing Guardrails in LangChain agents using the middleware system.

#### 📚 Topics Covered
- What are Guardrails & Why do they matter?
- Two approaches: Deterministic vs Model-based
- Built-in: PII Detection Middleware
- Built-in: Human-in-the-Loop Middleware
- Custom: Before-Agent Guardrail (input filtering)
- Custom: After-Agent Guardrail (output safety)
- Layered / Combined Guardrails
- Real-World Use Case: Healthcare Chatbot

`📌 Docs Reference: https://docs.langchain.com/oss/python/langchain/guardrails`

In [ ]:
from dotenv import load_dotenv
import os
from getpass import getpass

load_dotenv()
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

#### Section 1: What are Guardrails?
Guardrails help you build safe, compliant AI applications by validating and filtering content at key points in your agent's execution.

They are implemented as middleware that intercepts execution:

- Before the agent starts (input guardrails)
- After it completes (output guardrails)
- Around model and tool calls

Common Use Cases:
| Use Case                     | Example                                      |
|------------------------------|----------------------------------------------|
| PII leakage prevention       | Redact emails/credit cards before logging    |
| Prompt injection blocking    | Detect adversarial inputs                    |
| Harmful content filtering    | Block dangerous requests                     |
| Business rule enforcement    | Require approval for financial ops           |
| Output quality validation    | Ensure response meets safety standards       |

#### Section 2: Two Approaches to Guardrails
Deterministic Guardrails
- Rule-based: regex, keyword matching, explicit checks
- ✅ Fast, predictable, cost-effective
- ❌ May miss nuanced violations

Model-Based Guardrails
- Uses LLMs/classifiers for semantic understanding
- ✅ Catches subtle/nuanced issues
- ❌ Slower and more expensive

In [2]:
import re

# deterministic approach
def deterministic_guardrail(text: str) -> bool:
    """returns true if content is blocked"""
    banned_keywords = ["hack", "exploit", "malware", "bomb"]
    return any(kw in text.lower() for kw in banned_keywords)

test_inputs = ["How do I hack into a database ?", "what is the capital of france ?"]

print(f"=== Deterministic Guardrail ===")
for inp in test_inputs:
    blocked = deterministic_guardrail(inp)
    status = "BLOCKED" if blocked else "ALLOWED"
    print(f"{status}: {inp}")

=== Deterministic Guardrail ===
BLOCKED: How do I hack into a database ?
ALLOWED: what is the capital of france ?


In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    temperature=0
)

# model based approach
def model_based_guardrail(text: str) -> str:
    """Uses an LLM to evaluate content safely, returns SAFE or UNSAFE"""
    prompt = f"""Is the following user input safe to process? 
Reply with only 'SAFE' or 'UNSAFE'.

Input: {text}"""
    result = model.invoke([{"role": "user", "content": prompt}])
    return result.content.strip()

print(f"=== Model Baed Guardrail ===")
for inp in test_inputs:
    verdict = model_based_guardrail(inp)
    status = "UNSAFE" if "UNSAFE" in verdict else "SAFE"
    print(f"{status}: {inp}")

=== Model Baed Guardrail ===
UNSAFE: How do I hack into a database ?
SAFE: what is the capital of france ?


#### Section 3: Built-in Guardrail — PII Detection Middleware
LangChain provides built-in PIIMiddleware for detecting and handling Personally Identifiable Information (PII).

##### Supported PII Types:
| Type          | Example                              |
|---------------|--------------------------------------|
| email         | user@example.com                     |
| credit_card   | 5105-1051-0510-5100                  |
| ip            | 192.168.1.1                          |
| mac_address   | 00:1A:2B:3C:4D:5E                    |
| url           | https://secret-site.com              |

##### Strategies:
| Strategy | Result               |
|----------|----------------------|
| redact   | [REDACTED_EMAIL]     |
| mask     | ****-****-****-1234  |
| hash     | a8f5f167...          |
| block    | Raises an exception  |

In [16]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain_core.tools import tool
from langchain.chat_models import init_chat_model


groq_model = init_chat_model("groq:qwen/qwen3.8-27b")


@tool
def customer_lookup(query: str) -> str:
    """look up customer information"""
    return f"Customer record found for query: {query}"

agent = create_agent(
    model=model,
    tools=[customer_lookup],
    middleware=[
        # redact emails in user input before sending to model
        PIIMiddleware(
            "email",
            strategy="redact",
            apply_to_input=True
        ),
        # mask credit cards in user input
        PIIMiddleware(
            "credit_card",
            strategy="mask",
            apply_to_input=True
        ),
        # block API keys - raise error if detected
        PIIMiddleware(
            "api_keys",
            detector=r"sk-[a-zA-Z0-9]{32}",
            strategy="block",
            apply_to_input=True
        )
    ]
)
print("Agent with PII middleware created successfully")

Agent with PII middleware created successfully


In [17]:
result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "My email is jade.smith@example.com and my card is 4142-4423-1454-4766. Do you my information?"
    }]
})
print("=== Agent Response ===")
print(result["messages"][-1].content)

=== Agent Response ===
Yes, I found your customer record. What would you like to do with it?


In [ ]:
result
# somehow this google model is not working for card information

{'messages': [HumanMessage(content='My email is [REDACTED_EMAIL] and my card is 4142-4423-1454-4766. Do you my information?', additional_kwargs={}, response_metadata={}, id='e7f2afa0-5ce7-4cd6-97a3-f93b019b4340'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'customer_lookup', 'arguments': '{"query": "email: [REDACTED_EMAIL] and card: 4142-4423-1454-4766"}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0721a-ad3d-7473-9fcd-fe59e11c4d20-0', tool_calls=[{'name': 'customer_lookup', 'args': {'query': 'email: [REDACTED_EMAIL] and card: 4142-4423-1454-4766'}, 'id': '1e557a3b-e258-4fe1-82ca-f71dd802d0a3', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 77, 'output_tokens': 46, 'total_tokens': 123, 'input_token_details': {'cache_read': 0}}),
  ToolMessage(content='Customer record found for query: email: [REDACTED_EMAIL] and card: 

In [19]:
# test api key blocking
try:
    result = agent.invoke({
        "messages": [{
            "role": "user",
            "content": "Here is my key: sk-abcdefghijklmnopqrstuvwxyz123456"
        }]
    })
    
except Exception as e:
    print(f"🚫 Blocked as expected: {e}")

🚫 Blocked as expected: Detected 1 instance(s) of api_keys in text content


#### Section 4: Built-in Guardrail — Human-in-the-Loop Middleware
Pauses agent execution before sensitive operations and waits for human approval.

Best for:
- Financial transactions
- Sending emails to external parties
- Deleting production data
- Any operation with significant business impact

<b>Key requirement: A checkpointer for state persistence across interrupts.</b>

In [25]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from langchain_core.tools import tool


@tool
def search_web(query: str) -> str:
    """Search the web for information."""
    return f"Search results for: {query}"

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email to a recipient."""
    return f"Email sent to {to} with subject: {subject}"

@tool
def delete_records(table: str, condition: str) -> str:
    """Delete records from the database."""
    return f"Deleted records from {table} where {condition}"

hitl_agent = create_agent(
    model=model,
    tools=[search_web, send_email, delete_records],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email": True, # require approval
                "delete_record": True, # require approval
                "search_web": False # auto approve
            }
        )
    ],
    checkpointer=InMemorySaver() # require for state persistence
)

print("Human In The Loop agent created")

Human In The Loop agent created


In [27]:
# Step 1: Invoke — agent will pause before send_email
config = {"configurable": {"thread_id": "session_001"}}

result = hitl_agent.invoke(
    {"messages": [{"role": "user", "content": "Send an email to team@company.com about the Q4 results, decide subject body yourself"}]},
    config=config
)

print("=== Agent paused — awaiting human approval ===")
print(result)

=== Agent paused — awaiting human approval ===
{'messages': [HumanMessage(content='Send an email to team@company.com about the Q4 results', additional_kwargs={}, response_metadata={}, id='aa261df9-7df0-4888-bb0f-8e6bba8b2027'), AIMessage(content='Sure, I can help with that. What would you like the subject and body of the email to be?', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a07221-dc08-7fd2-9091-73742d705e11-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 163, 'output_tokens': 22, 'total_tokens': 185, 'input_token_details': {'cache_read': 0}}), HumanMessage(content='Send an email to team@company.com about the Q4 results, decide subject body yourself', additional_kwargs={}, response_metadata={}, id='f1e3600b-9b6c-4df9-9f76-aac3d3f82457'), AIMessage(content='', additional_kwargs={'function_call': {'name': 'send_email', 'argumen

In [28]:
# Step 2: Human reviews and APPROVES
approved_result = hitl_agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config   # Same thread_id resumes the paused session
)

print("=== Approved! Final response ===")
print(approved_result["messages"][-1].content)

=== Approved! Final response ===
I've sent the email to team@company.com with the subject "Q4 Results" and the body "The Q4 results are in. Please review them at your earliest convenience."


In [31]:
# Step 3: Alternative — Human REJECTS
config2 = {"configurable": {"thread_id": "session_003"}}

hitl_agent.invoke(
    {"messages": [{"role": "user", "content": "Delete all records from the users table"}]},
    config=config2
)

rejected_result = hitl_agent.invoke(
    Command(resume={"decisions": [{"type": "reject", "reason": "Too risky, needs DBA review"}]}),
    config=config2
)

print("=== Rejected! Final response ===")
print(rejected_result["messages"][-1].content)

=== Rejected! Final response ===
I cannot delete all records from the users table without a condition. Please provide a condition to delete specific records.


#### Section 5: Custom Guardrail — Before-Agent Hook (Input Filter)
Use `before_agent()` to validate or block requests before any LLM processing begins.

Best for:
- Keyword/content filtering
- Authentication checks
- Rate limiting
- Blocking specific categories of requests

In [35]:
from typing import Any
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langgraph.runtime import Runtime
from langchain.agents import create_agent
from langchain_core.tools import tool

class ContentFilterMiddleware(AgentMiddleware):
    """
    Deterministic guardrail: Block requests containing banned keywords.
    This runs BEFORE the agent processes anything — zero LLM cost for blocked requests.
    """
    def __init__(self, banned_keywords: list[str]):
        super().__init__()
        self.banned_keywords = [kw.lower() for kw in banned_keywords]
        
    @hook_config(can_jump_to=["end"])
    def before_agent(self, state, runtime):
        if not state["messages"]:
            return None
        
        first_message = state["messages"][0]
        if first_message.type != "human":
            return None
        
        content = first_message.content.lower()
        
        for keyword in self.banned_keywords:
            if keyword in content:
                print(f"BLocked - Keyword detected: {keyword}")
                return {
                    "messages": [{
                        "role": "assistant",
                        "content": (
                            "I can not process request containing inappropriate content."
                            "Please rephrase your request"
                        )
                    }],
                    "jumpt_to": "end"
                }
        return None
    
@tool
def search_tool(query: str) -> str:
    """Search for information."""
    return f"Results for: {query}"


# Create agent with content filter
filtered_agent = create_agent(
    model="google_genai:gemini-2.5-flash",
    tools=[search_tool],
    middleware=[
        ContentFilterMiddleware(
            banned_keywords=["hack", "exploit", "malware", "jailbreak", "bypass"]
        ),
    ],
)

print("Content filter agent created!")

Content filter agent created!


In [36]:
# Test 1: Safe request — should pass through
result = filtered_agent.invoke({
    "messages": [{"role": "user", "content": "What is machine learning?"}]
})
print("✅ Safe request response:")
print(result["messages"][-1].content)

✅ Safe request response:
[{'type': 'text', 'text': 'Machine learning is a subset of artificial intelligence (AI) that enables systems to learn from data, identify patterns, and make decisions with minimal human intervention. Instead of being explicitly programmed for every task, machine learning algorithms are trained on large datasets, allowing them to improve their performance over time as they are exposed to more data. This field encompasses various techniques, including supervised learning, unsupervised learning, and reinforcement learning, and is used in a wide range of applications such as image recognition, natural language processing, recommendation systems, and predictive analytics.', 'extras': {'signature': 'CqEJARFNMg9kEfAcUkzwnye2L2PII5HyI4bvAMNYnuTmympTvfNC9rFKMp2JiqJdhUBSsDB/b6JuwnTVNN/caqDBGh5sp7/+gNXcgqzTNF/WoZXmRfnv/TAVGtZCCFWbQox/AejGsLfQ5dD8Roa7q2CegzUo1gy7uc0h3w0EPQweMdrXe30Iqn+6lNQY7V+zPzTFHdtDsov6m3M6m8OPhhf2GuNlQLMa7l3jCzisLm43Ab7/GkA0MU6BGIEOXNvdanNFPBz4+IePEG2F

In [37]:
# Test 2: Unsafe request — should be blocked
result = filtered_agent.invoke({
    "messages": [{"role": "user", "content": "How do I hack into a server?"}]
})
print("🚫 Unsafe request response:")
print(result["messages"][-1].content)

BLocked - Keyword detected: hack
🚫 Unsafe request response:
.


#### Section 6: Custom Guardrail — After-Agent Hook (Output Safety)
Use after_agent() to validate the final agent response before the user sees it.

Best for:
- Model-based safety evaluation of outputs
- Compliance scanning (e.g. legal, medical, financial disclaimers)
- Quality validation
- Removing sensitive info that slipped through

In [47]:
from typing import Any
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langgraph.runtime import Runtime
from langchain_core.messages import AIMessage
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain_core.tools import tool


class SafetyGuardrailMiddleware(AgentMiddleware):
    """
    Model-based guardrail: Use an LLM to evaluate response safety.
    Runs AFTER the agent produces a response, before it reaches the user.
    """

    def __init__(self):
        super().__init__()
        # Use a smaller, cheaper model for the safety check
        self.safety_model = model

    @hook_config(can_jump_to=["end"])
    def after_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if not state["messages"]:
            return None

        last_message = state["messages"][-1]
        if not isinstance(last_message, AIMessage):
            return None

        # Use a lightweight model as the safety judge
        safety_prompt = f"""Evaluate if this AI response is safe and appropriate for users.
Respond with only 'SAFE' or 'UNSAFE'.

Response to evaluate:
{last_message.content}"""

        result = self.safety_model.invoke([{"role": "user", "content": safety_prompt}])

        if "UNSAFE" in result.content.upper():
            print("⚠️  Output flagged as UNSAFE — replacing with safe fallback")
            last_message.content = (
                "I'm unable to provide that response. "
                "Please rephrase your request or contact support."
            )

        return None
    

@tool
def general_tool(query: str) -> str:
    """A general purpose tool."""
    return f"Tool result: {query}"


safe_agent = create_agent(
    model="google_genai:gemini-2.5-flash-lite",
    tools=[general_tool], 
    middleware=[SafetyGuardrailMiddleware()],
)

print("Output safety agent created!")

Output safety agent created!


In [48]:
# Test output safety check
result = safe_agent.invoke({
    "messages": [{"role": "user", "content": "What is the weather like today?"}]
})
print("Response:")
print(result["messages"][-1].content)

Response:
I need to know your location to tell you the weather. Could you please tell me where you are?


#### Section 7: Layered / Combined Guardrails
Stack multiple guardrails in the middleware=[] array. They execute in order, building layered protection.

```
User Input
    ↓
[Layer 1] ContentFilterMiddleware    ← Deterministic input filter
    ↓
[Layer 2] PIIMiddleware (input)      ← PII redaction on input
    ↓
[Layer 3] HumanInTheLoopMiddleware   ← Approval for sensitive tools
    ↓
[Layer 4] PIIMiddleware (output)     ← PII redaction on output
    ↓
[Layer 5] SafetyGuardrailMiddleware  ← Model-based output safety
    ↓
User Response
```

In [49]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware, HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.tools import tool

@tool
def search_tool(query: str) -> str:
    """Search for information."""
    return f"Search results: {query}"

@tool
def send_email_tool(to: str, body: str) -> str:
    """Send an email."""
    return f"Email sent to {to}"

# full layered guardrail stack
production_agent = create_agent(
    model=model,
    tools=[search_tool, send_email_tool],
    middleware=[
        #layer 1: deterministic input filter
        ContentFilterMiddleware(banned_keywords=["hack", "exploit", "malware"]),
        #layer 2: PII redaction on input
        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),
        #layer 3: Human approval for sensitive tools
        HumanInTheLoopMiddleware(interrupt_on={"send_email_tool": True, "search_tool": False}),
        #layer 4: PII redaction on output
        PIIMiddleware("email", strategy="redact", apply_to_output=True),
        #layer 5: model based output safety
        SafetyGuardrailMiddleware()
    ],
    checkpointer=InMemorySaver()
)

print("Production agent with 5 layer guardrails created")

Production agent with 5 layer guardrails created


#### Section 8: Real-World Use Case — Healthcare Chatbot
A healthcare chatbot that:

- Blocks off-topic or harmful requests
- Redacts patient PII (emails, credit card numbers)
- Requires human approval before booking appointments
- Validates that outputs are medically appropriate

In [55]:
from typing import Any
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langchain.agents.middleware import PIIMiddleware, HumanInTheLoopMiddleware
from langgraph.runtime import Runtime
from langchain.agents import create_agent
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langchain_openai import ChatOpenAI
from langchain_core.messages import AIMessage
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)


# --- Healthcare-specific content filter ---
class HealthcareSafetyFilter(AgentMiddleware):
    """Block non-medical or harmful requests in a healthcare context."""

    BLOCKED_TOPICS = ["drug synthesis", "self-harm", "suicide method", "weapon", "hack"]

    @hook_config(can_jump_to=["end"])
    def before_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if not state["messages"]:
            return None

        first_msg = state["messages"][0]
        if first_msg.type != "human":
            return None

        content = first_msg.content.lower()
        for topic in self.BLOCKED_TOPICS:
            if topic in content:
                return {
                    "messages": [{
                        "role": "assistant",
                        "content": (
                            "I'm a healthcare assistant and can only help with "
                            "medical questions, appointments, and health information. "
                            "If you're in crisis, please call 112 or your local emergency number."
                        )
                    }],
                    "jump_to": "end"
                }
        return None
    

# --- Medical output validator ---
class MedicalOutputValidator(AgentMiddleware):
    """Ensure all responses include appropriate medical disclaimers."""

    DISCLAIMER = "\n\n⚕️ *This is general health information, not medical advice. Please consult a qualified healthcare professional.*"

    @hook_config(can_jump_to=["end"])
    def after_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if not state["messages"]:
            return None

        last_message = state["messages"][-1]
        if not isinstance(last_message, AIMessage):
            return None

        # Add disclaimer if not already present
        if "medical advice" not in last_message.content.lower():
            last_message.content += self.DISCLAIMER

        return None
    

# --- Healthcare tools ---
@tool
def search_symptoms(symptoms: str) -> str:
    """Search for information about medical symptoms."""
    return f"Symptom information for: {symptoms}. Please consult a doctor for diagnosis."

@tool
def book_appointment(patient_name: str, date: str, doctor: str) -> str:
    """Book a medical appointment."""
    return f"Appointment booked for {patient_name} with Dr. {doctor} on {date}"

@tool
def get_medication_info(medication: str) -> str:
    """Get information about a medication."""
    return f"General info about {medication}. Always follow your doctor's prescription."


# build healthcare chatbot
healthcare_bot = create_agent(
    model=model,
    tools=[search_symptoms, book_appointment, get_medication_info],
    middleware=[
        HealthcareSafetyFilter(),
        PIIMiddleware("email", strategy="redact", apply_to_input=True),
        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),
        HumanInTheLoopMiddleware(
            interrupt_on={
                "book_appointment": True,
                "get_medication_info": False,
                "search_symptoms": False
            }
        ),
        MedicalOutputValidator()
    ],
    checkpointer=InMemorySaver(),
    system_prompt=(
        "You are a helpful healthcare assistant. "
        "You can search for symptoms, medication information, and help book appointments. "
        "Always be empathetic and remind users to consult a doctor for diagnosis."
    )
)

print("🏥 Healthcare chatbot with full guardrail stack created!")

🏥 Healthcare chatbot with full guardrail stack created!


In [56]:
# Test 1: Safe medical query
config_t1 = {"configurable": {"thread_id": "healthcare_session_t1"}}

result = healthcare_bot.invoke(
    {"messages": [{"role": "user", "content": "What are symptoms of Type 2 Diabetes?"}]},
    config=config_t1
)

result

{'messages': [HumanMessage(content='What are symptoms of Type 2 Diabetes?', additional_kwargs={}, response_metadata={}, id='af9508ad-5a9e-4e3c-b06a-2fec8d6d2a45'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'search_symptoms', 'arguments': '{"symptoms": "Type 2 Diabetes"}'}, '__gemini_function_call_thought_signatures__': {'3cb65d46-08a1-4cc0-9a77-1926725b445b': 'CvoBARFNMg8jXA5zmjHiHa3/BafTf0sA3IQipgqyNrBtso2ZA8rV4rG1EOmSsS8EuCpu6H+uW3cUlzJg/+bcpqHcC5bckiQAgbhY97+Iv5RSgzKqdpnaSmYZVDo2ZFwAi62vwXQCoCa8R4r/z0WrPjWklTmevYl3xpuOspqbrZGUJUMBEcy+MSuuJBz4FqAS/Vd8YjeNCO67ThIPiuflu+qJqD7Ib7lBwqZ9fRD/mrvqlj4bkS2V2H7w9H/qN2BmZR/JYZIN9Ct+Zuu0q/QOzIKUCTPQmQwqsj7iWx8/8w9d/VzS2nRF2hBoojZO8A7S5YWnAuOE5xd0tclmZg=='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a076c6-196c-7063-bec7-7547658461fd-0', tool_calls=[{'name': 'search_symptoms', 'args': {'symptoms': 'Type 2 Diabetes

In [57]:
result["messages"][-1].content

"Type 2 Diabetes symptoms can include increased thirst, frequent urination, increased hunger, fatigue, blurred vision, slow-healing sores, and frequent infections. It's important to remember that I am an AI and cannot provide a diagnosis. Please consult a doctor for accurate diagnosis and treatment.\n\n⚕️ *This is general health information, not medical advice. Please consult a qualified healthcare professional.*"

In [58]:
# Test 2: Query with PII (email gets redacted)
result = healthcare_bot.invoke({
    "messages": [{
        "role": "user",
        "content": "My email is patient123@gmail.com. What can I take for a headache?"
    }]},
    config=config_t1
)
print("=== PII Redaction Test ===")
print(result["messages"][-1].content)

=== PII Redaction Test ===
I cannot give medical advice or recommend specific medications. It's always best to consult a healthcare professional for any health concerns, including headaches. They can provide an accurate diagnosis and recommend the most appropriate treatment for you.


In [59]:
# Test 3: Off-topic / harmful request — gets blocked
result = healthcare_bot.invoke({
    "messages": [{"role": "user", "content": "How do I synthesize drugs at home?"}]
},
 config=config_t1)
print("=== Blocked Request ===")
print(result["messages"][-1].content)

=== Blocked Request ===
I cannot provide information or guidance on synthesizing drugs at home. Attempting to synthesize drugs can be extremely dangerous and illegal. It can lead to serious health risks, including poisoning, explosions, and other severe injuries, not to mention legal consequences.

If you have questions about medications or health concerns, please consult a qualified healthcare professional or pharmacist. They can provide you with accurate and safe information.

⚕️ *This is general health information, not medical advice. Please consult a qualified healthcare professional.*


In [60]:
# Test 4: Appointment booking — requires human approval
config = {"configurable": {"thread_id": "healthcare_session_001"}}

result = healthcare_bot.invoke(
    {"messages": [{"role": "user", "content": "Book me an appointment with Dr. Sharma on March 15"}]},
    config=config
)
print("=== Appointment Booking — Awaiting Approval ===")
print(result)

# Approve
from langgraph.types import Command
approved = healthcare_bot.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config
)
print("\n=== After Approval ===")
print(approved["messages"][-1].content)

=== Appointment Booking — Awaiting Approval ===
{'messages': [HumanMessage(content='Book me an appointment with Dr. Sharma on March 15', additional_kwargs={}, response_metadata={}, id='27afcef4-2f66-4b11-b80a-437201e2cbb5'), AIMessage(content='I can help you book an appointment with Dr. Sharma for March 15th. What is your name?\n\n⚕️ *This is general health information, not medical advice. Please consult a qualified healthcare professional.*', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a076c7-3d93-7c91-8c67-8d36ed809aa6-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 196, 'output_tokens': 23, 'total_tokens': 219, 'input_token_details': {'cache_read': 0}})]}

=== After Approval ===
I can help you book an appointment with Dr. Sharma for March 15th. What is your name?


#### Summary

| Guardrail Type       | Hook             | When it Runs            | Best For                          |
|----------------------|------------------|--------------------------|-----------------------------------|
| PII Middleware       | Input/Output     | Around model calls       | Data privacy, compliance          |
| Human-in-the-Loop    | Tool level       | Before sensitive tools   | High-stakes decisions             |
| Content Filter       | before_agent     | Start of invocation      | Blocking bad inputs early         |
| Safety Validator     | after_agent      | End of invocation        | Output quality/safety             |
| Custom Logic         | Any hook         | Anywhere                 | Any business rule                 |

##### Key Takeaways
1. **Guardrails = Middleware** — implement them via the `middleware=[]` parameter in `create_agent()`
2. **Layer your guardrails** — defense in depth is best practice
3. **Deterministic first, model-based second** — use cheap rule-based checks early to avoid expensive LLM calls
4. **Human-in-the-Loop requires a checkpointer** — use `InMemorySaver` for dev, persistent store for production
5. **Custom middleware** gives you full control via `before_agent()` and `after_agent()` hooks

##### Additional Resources
- [LangChain Guardrails Docs](https://docs.langchain.com/oss/python/langchain/guardrails)
- [Middleware Docs](https://docs.langchain.com/oss/python/langchain/middleware/overview)
- [Human-in-the-Loop Docs](https://docs.langchain.com/oss/python/langchain/human-in-the-loop)
- [LangSmith for Observability](https://docs.langchain.com/oss/python/langchain/observability)

